<a href="https://colab.research.google.com/github/jetsonmom/6.23_automobility_lesson/blob/main/0812_etriai_segformer_class8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1단계


In [ ]:
# -h 옵션은 파일 크기를 사람이 보기 쉽게 (예: 520M) 보여줍니다.
!ls -lh /content/sample_data/
!rm "/content/sample_data/MonoCameraSemanticSegmentation.zip"



In [ ]:
# 1. 불필요한 중복 파일 삭제 (좋은 습관입니다)
#!rm "/content/sample_data/MonoCameraSemanticSegmentation (2).zip"

# 2. 메인 ZIP 파일의 압축을 /workspace 폴더에 해제
!unzip -q /content/sample_data/MonoCameraSemanticSegmentation.zip -d /content/sample_data/

# 3. 압축 해제 결과 확인
!ls -l /content/sample_data/MonoCameraSemanticSegmentation/

In [ ]:
# 1단계: 환경 설정 및 데이터 전처리 (전체 데이터 사용)
# JSON 라벨 파일을 마스크 이미지로 변환

# 필요한 라이브러리 설치
!pip install -q opencv-python numpy

# 기본 라이브러리 임포트
import os
import json
import numpy as np
import cv2

print("라이브러리 로드 완료!")

# 압축 파일 해제
print("압축 파일 해제 중...")

# MonoCameraSemanticSegmentation.zip 해제
if os.path.exists('/content/sample_data/MonoCameraSemanticSegmentation(2).zip'):
    !cd /content/sample_data && unzip -q MonoCameraSemanticSegmentation.zip
    print("✅ MonoCameraSemanticSegmentation.zip 해제 완료!")

# labels.zip 해제
#if os.path.exists('/content/sample_data/labels.zip'):
#    !cd /content/sample_data && unzip -q labels.zip
#    print("✅ labels.zip 해제 완료!")

# 🔍 실제 폴더 구조 확인
print("\n🔍 실제 폴더 구조 확인:")
print("sample_data 폴더 내용:")
!ls -la /content/sample_data/

# 🎯 실제 이미지 폴더 찾기
possible_image_paths = [
    '/content/sample_data/JPEGImages_mosaic',  # 바로 sample_data 아래
    '/content/sample_data/MonoCameraSemanticSegmentation/JPEGImages_mosaic',
    '/content/sample_data/MonoCameraSemanticSegmentation/JPEGImages',
]

IMAGE_DIR = None
for path in possible_image_paths:
    if os.path.exists(path):
        IMAGE_DIR = path
        print(f"✅ 이미지 폴더 찾음: {path}")
        # 몇 개 파일이 있는지 확인
        png_files = [f for f in os.listdir(path) if f.endswith('.png')]
        print(f"   📁 PNG 파일 수: {len(png_files)}개")
        break

if IMAGE_DIR is None:
    print("❌ 이미지 폴더를 찾을 수 없습니다!")
    print("🔍 sample_data 하위 폴더 전체 탐색:")
    for item in os.listdir('/content/sample_data/'):
        item_path = os.path.join('/content/sample_data/', item)
        if os.path.isdir(item_path):
            print(f"  📁 {item}/")
            try:
                files = os.listdir(item_path)
                png_count = len([f for f in files if f.endswith('.png')])
                if png_count > 0:
                    print(f"     🖼️ PNG 파일 {png_count}개 발견!")
                    IMAGE_DIR = item_path
            except:
                pass

# 경로 설정
LABEL_DIR = '/content/sample_data/labels'
MASK_DIR = '/content/sample_data/masks_final_selected' # 마스크 저장 폴더 이름 변경

# 새롭게 정의된 클래스 맵
class_map = {
    "background": 0,
    "bus": 1,
    "car": 2,
    "person": 3,
    "whdot": 4,
    "whsol": 5,
    "yesol": 6,
    "crosswalk": 7,
    "traffic light": 8,
    "traffic sign": 9,
    "pole": 10
}

print("경로 설정 완료!")
print(f"이미지 폴더: {IMAGE_DIR}")
print(f"라벨 폴더: {LABEL_DIR}")
print(f"마스크 저장 폴더: {MASK_DIR}")
print(f"사용할 클래스 매핑: {class_map}")


# 전체 데이터 현황 파악
print("\n📊 전체 데이터 현황 파악 중...")

if os.path.exists(IMAGE_DIR):
    all_images = [f for f in os.listdir(IMAGE_DIR) if f.endswith('.png')]
    daeduk_images = [f for f in all_images if f.startswith('Daeduk')]
    sangam_images = [f for f in all_images if f.startswith('SangamDMC')]

    print(f"🖼️ 이미지 파일 현황:")
    print(f"  - Daeduk: {len(daeduk_images)}개")
    print(f"  - SangamDMC: {len(sangam_images)}개")
    print(f"  - 총 이미지: {len(all_images)}개")

if os.path.exists(LABEL_DIR):
    all_labels = [f for f in os.listdir(LABEL_DIR) if f.endswith('.json')]
    daeduk_labels = [f for f in all_labels if f.startswith('Daeduk')]
    sangam_labels = [f for f in all_labels if f.startswith('SangamDMC')]

    print(f"🏷️ 라벨 파일 현황:")
    print(f"  - Daeduk: {len(daeduk_labels)}개")
    print(f"  - SangamDMC: {len(sangam_labels)}개")
    print(f"  - 총 라벨: {len(all_labels)}개")

# 매칭 가능한 데이터 분석
print(f"\n🎯 매칭 분석:")

# 변수 초기화
matched_basenames = []
daeduk_matched = []
sangam_matched = []

if IMAGE_DIR and os.path.exists(IMAGE_DIR) and os.path.exists(LABEL_DIR) and len(all_images) > 0 and len(all_labels) > 0:
    # 이미지 베이스명들
    image_basenames = [f.replace('_leftImg8bit.png', '') for f in all_images]
    # 라벨 베이스명들
    label_basenames = [f.replace('_gtFine_polygons.json', '') for f in all_labels]

    # 매칭되는 것들 찾기
    matched_basenames = list(set(image_basenames) & set(label_basenames))

    daeduk_matched = [name for name in matched_basenames if name.startswith('Daeduk')]
    sangam_matched = [name for name in matched_basenames if name.startswith('SangamDMC')]

    print(f"  - Daeduk 매칭: {len(daeduk_matched)}개")
    print(f"  - SangamDMC 매칭: {len(sangam_matched)}개")
    print(f"  - 총 사용가능: {len(matched_basenames)}개")
else:
    if not IMAGE_DIR or not os.path.exists(IMAGE_DIR):
        print("❌ 이미지 폴더를 찾을 수 없습니다!")
    elif not os.path.exists(LABEL_DIR):
        print("❌ 라벨 폴더를 찾을 수 없습니다!")
    elif len(all_images) == 0:
        print("❌ 이미지 파일이 없습니다!")
    elif len(all_labels) == 0:
        print("❌ 라벨 파일이 없습니다!")

print(f"🎯 처리할 총 데이터: {len(matched_basenames)}개")

# 마스크 이미지 생성
print("\n🚀 전체 데이터로 마스크 이미지 생성 시작...")
os.makedirs(MASK_DIR, exist_ok=True)

if len(matched_basenames) == 0:
    print("❌ 처리할 데이터가 없습니다! 경로를 확인해주세요.")
    print("\n✨ 1단계 완료 (데이터 없음)")
else:
    processed_count = 0
    error_count = 0

    # 매칭된 데이터만 처리
    for i, base_name in enumerate(matched_basenames):
        # 파일 경로 구성
        json_file = f"{base_name}_gtFine_polygons.json"
        image_file = f"{base_name}_leftImg8bit.png"

        label_path = os.path.join(LABEL_DIR, json_file)
        image_path = os.path.join(IMAGE_DIR, image_file)

        # 이미지 로드
        image = cv2.imread(image_path)
        if image is None:
            print(f"  ❌ {i+1}/{len(matched_basenames)} 에러: 이미지 로드 실패 ({image_file})")
            error_count += 1
            continue

        # 이미지 크기 가져오기
        height, width, _ = image.shape

        # JSON 라벨 파일 읽기
        try:
            with open(label_path, 'r') as f:
                data = json.load(f)
        except:
            print(f"  ❌ {i+1}/{len(matched_basenames)} 에러: JSON 파일 읽기 실패 ({json_file})")
            error_count += 1
            continue

        # 빈 마스크 생성 (class_map의 마지막 값 + 1 만큼의 채널을 가질 수 있도록 수정)
        # 각 클래스별 마스크를 별도로 저장하려면 각 클래스별로 0으로 초기화된 마스크를 생성해야 합니다.
        # 여기서는 모든 선택된 객체를 단일 마스크에 다른 값으로 표시하는 방식으로 구현합니다.
        mask = np.zeros((height, width), dtype=np.uint8)


        # 선택된 객체 및 차선 폴리곤을 마스크에 그리기
        found_labels_count = 0
        for obj in data['objects']:
            label = obj['label']
            points = np.array(obj['polygon'], dtype=np.int32)

            # 🎯 사용자가 선택한 라벨들
            selected_labels = [
                'bus',
                'car',
                'person',
                'whdot',
                'whsol',
                'yesol',
                'crosswalk',
                'traffic light',
                'traffic sign',
                'pole'
            ]

            if label in selected_labels:
                # class_map에서 해당 라벨의 값으로 마스크에 그리기
                if label in class_map:
                    cv2.fillPoly(mask, [points], color=class_map[label])
                    found_labels_count += 1


        # 마스크 저장 (새로운 폴더에 저장)
        mask_file_name = image_file.replace('.png', '_mask.png')
        mask_save_path = os.path.join(MASK_DIR, mask_file_name)
        cv2.imwrite(mask_save_path, mask)

        processed_count += 1
        area_type = "🟦 Daeduk" if base_name.startswith('Daeduk') else "🟩 SangamDMC"
        print(f"  ✅ {i+1}/{len(matched_basenames)} {area_type}: {mask_file_name} (선택된 라벨 {found_labels_count}개)")

    print(f"\n🎉 전체 데이터 마스크 생성 완료!")
    print(f"✅ 성공: {processed_count}개 (Daeduk + SangamDMC)")
    print(f"❌ 실패: {error_count}개")
    print(f"📁 저장 위치: {MASK_DIR}")

    # 생성된 파일 확인
    mask_files = [f for f in os.listdir(MASK_DIR) if f.endswith('_mask.png')]
    print(f"📊 생성된 마스크 파일 수: {len(mask_files)}개")

    print("\n✨ 1단계 완료! 다음 단계로 진행하세요.")

🔍 분석 코드
 이 코드를 실행하면:
위치별 차선 분포 확인 (왼쪽 vs 오른쪽) 차선 타입별 크기 분석 불균형 정도 진단 해결책 제시

In [ ]:
# 차선 위치 및 클래스별 분석
# 왜 오른쪽 차선만 잘 인식되는지, 어떤 클래스가 있는지 분석

import json
import os
import cv2
import numpy as np
from collections import Counter, defaultdict

print("🔍 차선 위치 및 클래스별 분석 시작!")

# --- 경로 설정 ---
# ※ 환경에 맞게 경로를 확인해주세요.
LABEL_DIR = '/content/sample_data/labels'
# IMAGE_DIR 경로를 실제 압축 해제된 경로로 정확히 지정해야 합니다.
possible_image_paths = [
    '/content/sample_data/JPEGImages_mosaic',
    '/content/sample_data/MonoCameraSemanticSegmentation/JPEGImages_mosaic',
    '/content/sample_data/MonoCameraSemanticSegmentation/JPEGImages',
]
IMAGE_DIR = None
for path in possible_image_paths:
    if os.path.exists(path):
        IMAGE_DIR = path
        break

if not LABEL_DIR or not IMAGE_DIR or not os.path.exists(LABEL_DIR) or not os.path.exists(IMAGE_DIR):
    print(f"❌ 경로 오류: 다음 폴더가 존재하는지 확인하세요.")
    print(f"  - 라벨 폴더: {LABEL_DIR}")
    print(f"  - 이미지 폴더: {IMAGE_DIR or '경로를 찾을 수 없음'}")
    # Google Colab 환경에서 실행하는 경우, 이전 단계의 코드 셀을 먼저 실행하여
    # 압축을 해제하고 파일 경로를 확인해야 합니다.
else:
    # --- 분석 설정 ---
    lane_labels = ['whdot', 'whsol', 'yesol', 'yedot', 'blsol', 'bldot', 'general road mark']
    lane_position_stats = {'left_side': Counter(), 'right_side': Counter(), 'center': Counter()}
    all_found_labels = set()

    print("📊 샘플 데이터 분석 중...")
    json_files = [f for f in os.listdir(LABEL_DIR) if f.endswith('.json')][:100] # 분석 샘플 수 증가

    for i, json_file in enumerate(json_files):
        if i > 0 and i % 20 == 0:
            print(f"  진행: {i}/{len(json_files)}")

        json_path = os.path.join(LABEL_DIR, json_file)
        image_name = json_file.replace('_gtFine_polygons.json', '_leftImg8bit.png')
        image_path = os.path.join(IMAGE_DIR, image_name)

        if not os.path.exists(image_path): continue
        image = cv2.imread(image_path)
        if image is None: continue

        height, width = image.shape[:2]

        with open(json_path, 'r') as f:
            data = json.load(f)

        for obj in data.get('objects', []):
            label = obj.get('label', '')
            if not label: continue
            all_found_labels.add(label)

            if label in lane_labels:
                points = np.array(obj['polygon'])
                center_x = np.mean(points[:, 0])

                if center_x < width * 0.4:
                    lane_position_stats['left_side'][label] += 1
                elif center_x > width * 0.6:
                    lane_position_stats['right_side'][label] += 1
                else:
                    lane_position_stats['center'][label] += 1

    print("✅ 분석 완료!")

    # --- 결과 출력 ---

    # 🔬 1. 데이터셋에서 발견된 모든 객체 클래스 목록 (세로 출력으로 수정)
    print("\n" + "="*50)
    print("🔬 데이터셋에서 발견된 모든 객체 클래스 목록")
    print("="*50)
    if all_found_labels:
        # 알파벳 순으로 정렬하여 보기 쉽게 만듦
        sorted_labels = sorted(list(all_found_labels))
        # 반복문을 사용하여 한 줄에 하나씩 출력
        for label in sorted_labels:
            print(f"  - {label}")
    else:
        print("  - 발견된 클래스가 없습니다.")

    # 📊 2. 차선 클래스의 위치별 분포
    print("\n" + "="*50)
    print(f"📊 차선 클래스의 위치별 분포 (샘플 {len(json_files)}개 기준)")
    print("="*50)

    total_by_position = {}
    for position in ['left_side', 'center', 'right_side']:
        total = sum(lane_position_stats[position].values())
        total_by_position[position] = total

        print(f"\n🔍 {position.upper()} ({total}개):")
        if not lane_position_stats[position]:
            print("  - 데이터 없음")
        else:
            for label, count in lane_position_stats[position].most_common():
                print(f"  - [클래스: {label}]: {count}개")

    # 📈 3. 위치별 총합 및 비율
    print("\n" + "="*50)
    print(f"📈 위치별 총합 및 비율")
    print("="*50)
    total_lanes_found = sum(total_by_position.values())
    if total_lanes_found > 0:
        for position, total in total_by_position.items():
            percentage = (total / total_lanes_found) * 100
            print(f"  - {position}: {total}개 ({percentage:.1f}%)")
    else:
        print("  - 분석된 차선 데이터가 없습니다.")

    # 🎯 4. 불균형 진단
    print("\n" + "="*50)
    print(f"🎯 불균형 진단")
    print("="*50)

    left_total = total_by_position.get('left_side', 0)
    right_total = total_by_position.get('right_side', 0)

    if right_total > left_total * 1.5:
        print(f"❌ 오른쪽 편향: 오른쪽 차선({right_total}개)이 왼쪽({left_total}개)보다 1.5배 이상 많습니다.")
        print(f"   ➡️ 해결책: 이미지 좌우 반전(Horizontal Flip)과 같은 데이터 증강 기법이 필요합니다.")
    elif left_total > right_total * 1.5:
        print(f"❌ 왼쪽 편향: 왼쪽 차선({left_total}개)이 오른쪽({right_total}개)보다 1.5배 이상 많습니다.")
    else:
        print(f"✅ 좌우 균형 양호: 왼쪽({left_total}개) vs 오른쪽({right_total}개)")

    print("\n✨ 차선 위치 및 클래스 분석 완료!")

특정 객체(bus, car, person)와 차선 관련 라벨(whdot, whsol, yesol, crosswalk, traffic light, traffic sign, pole)만 사용하여 마스크를 생성

class_map = {
    "background": 0,
    "lane": 1,          # 모든 차선 관련 라벨은 'lane' 클래스(ID 1)로 통합됩니다.
    "bus": 2,
    "car": 3,
    "person": 4,
    "crosswalk": 5,
    "traffic_light": 6,
    "traffic_sign": 7,
    "pole": 8,
}
target_labels = {
    # 차선 관련 라벨 -> 'lane' 클래스로 매핑
    'whdot': 'lane',
    'whsol': 'lane',
    'yesol': 'lane',
    'yedot': 'lane',
    'blsol': 'lane',
    'bldot': 'lane',
    'general road mark': 'lane',
    
 ### 기타 객체 라벨 -> 해당 클래스로 직접 매핑
    'bus': 'bus',
    'car': 'car',
    'person': 'person',
    'crosswalk': 'crosswalk',
    'traffic_light': 'traffic_light',
    'traffic_sign': 'traffic_sign',
    'pole': 'pole',

In [ ]:
# 1단계 (수정): 특정 클래스만 선택하여 다중 클래스 마스크 생성

# 기본 라이브러리 임포트
import os
import json
import numpy as np
import cv2
from collections import Counter

print("라이브러리 로드 완료!")

# --- 경로 설정 ---
# ※ 이전 단계에서 확인된 경로를 사용합니다.
LABEL_DIR = '/content/sample_data/labels'

# 실제 이미지 폴더 경로를 자동으로 찾습니다.
possible_image_paths = [
    '/content/sample_data/JPEGImages_mosaic',
    '/content/sample_data/MonoCameraSemanticSegmentation/JPEGImages_mosaic',
    '/content/sample_data/MonoCameraSemanticSegmentation/JPEGImages',
]
IMAGE_DIR = None
for path in possible_image_paths:
    if os.path.exists(path):
        IMAGE_DIR = path
        break

# 새 마스크를 저장할 폴더
MASK_DIR_MULTI = '/content/sample_data/masks_multi_class'
os.makedirs(MASK_DIR_MULTI, exist_ok=True)

# --- 클래스 정의 ---
# 🎯 모델이 최종적으로 학습할 클래스와 고유 ID를 정의합니다.
# 이 맵은 나중에 모델 학습 시 id2label, label2id로 사용됩니다.
class_map = {
    "background": 0,
    "lane": 1,          # 모든 차선 관련 라벨은 'lane' 클래스(ID 1)로 통합됩니다.
    "bus": 2,
    "car": 3,
    "person": 4,
    "crosswalk": 5,
    "traffic_light": 6,
    "traffic_sign": 7,
    "pole": 8,
}

# 🎯 JSON 파일의 라벨 중 어떤 것을 사용할지, 그리고 어떤 클래스로 매핑할지 정의합니다.
target_labels = {
    # 차선 관련 라벨 -> 'lane' 클래스로 매핑
    'whdot': 'lane',
    'whsol': 'lane',
    'yesol': 'lane',
    'yedot': 'lane',
    'blsol': 'lane',
    'bldot': 'lane',
    'general road mark': 'lane',

    # 기타 객체 라벨 -> 해당 클래스로 직접 매핑
    'bus': 'bus',
    'car': 'car',
    'person': 'person',
    'crosswalk': 'crosswalk',
    'traffic_light': 'traffic_light',
    'traffic_sign': 'traffic_sign',
    'pole': 'pole',
}

print("✅ 클래스 및 경로 설정 완료")
print(f"  - 최종 클래스 수: {len(class_map)}개")
print(f"  - 마스크 저장 위치: {MASK_DIR_MULTI}")

# --- 데이터 매칭 ---
if not IMAGE_DIR or not os.path.exists(IMAGE_DIR):
    print("❌ 이미지 폴더를 찾을 수 없습니다! 이전 단계의 압축 해제가 정상적으로 완료되었는지 확인하세요.")
else:
    all_images = [f for f in os.listdir(IMAGE_DIR) if f.endswith('.png')]
    all_labels = [f for f in os.listdir(LABEL_DIR) if f.endswith('.json')]

    image_basenames = {f.replace('_leftImg8bit.png', '') for f in all_images}
    label_basenames = {f.replace('_gtFine_polygons.json', '') for f in all_labels}

    matched_basenames = sorted(list(image_basenames & label_basenames))
    print(f"🎯 처리할 총 데이터: {len(matched_basenames)}개")

    # --- 다중 클래스 마스크 생성 시작 ---
    print("\n🚀 다중 클래스 마스크 생성 시작...")
    processed_count = 0
    error_count = 0
    class_counts = Counter() # 각 클래스가 몇 번 그려졌는지 카운트

    # 매칭된 데이터만 처리
    for i, base_name in enumerate(matched_basenames):
        json_file = f"{base_name}_gtFine_polygons.json"
        image_file = f"{base_name}_leftImg8bit.png"

        label_path = os.path.join(LABEL_DIR, json_file)
        image_path = os.path.join(IMAGE_DIR, image_file)

        image = cv2.imread(image_path)
        if image is None:
            error_count += 1
            continue

        height, width, _ = image.shape
        mask = np.zeros((height, width), dtype=np.uint8) # 0 (background)으로 초기화

        try:
            with open(label_path, 'r') as f:
                data = json.load(f)
        except Exception as e:
            error_count += 1
            continue

        # JSON 파일의 각 객체에 대해 반복
        for obj in data['objects']:
            label_from_json = obj['label']

            # 이 라벨이 우리가 타겟으로 하는 라벨인지 확인
            if label_from_json in target_labels:

                # 최종 클래스 이름 가져오기 (예: 'whdot' -> 'lane')
                target_class_name = target_labels[label_from_json]

                # 클래스 ID 가져오기 (예: 'lane' -> 1)
                class_id = class_map[target_class_name]

                # 폴리곤 좌표 가져오기
                points = np.array(obj['polygon'], dtype=np.int32)

                # 마스크에 해당 클래스 ID로 폴리곤 그리기
                cv2.fillPoly(mask, [points], color=class_id)

                # 통계용 카운트
                class_counts[target_class_name] += 1

        # 마스크 파일 저장
        mask_file_name = image_file.replace('.png', '_mask_multi.png')
        mask_save_path = os.path.join(MASK_DIR_MULTI, mask_file_name)
        cv2.imwrite(mask_save_path, mask)

        processed_count += 1
        if i % 100 == 0: # 100개마다 진행 상황 출력
             print(f"  ✅ {i+1}/{len(matched_basenames)}: {mask_file_name} 생성 완료")

    print(f"\n🎉 다중 클래스 마스크 생성 완료!")
    print(f"  - ✅ 성공: {processed_count}개")
    print(f"  - ❌ 실패: {error_count}개")
    print(f"  - 📁 저장 위치: {MASK_DIR_MULTI}")

    # 생성된 파일 수 확인
    mask_files = [f for f in os.listdir(MASK_DIR_MULTI) if f.endswith('_mask_multi.png')]
    print(f"  - 📊 생성된 마스크 파일 수: {len(mask_files)}개")

    print("\n📊 처리된 클래스별 객체 수:")
    if not class_counts:
        print("  - 처리된 객체가 없습니다.")
    else:
        for class_name, count in class_counts.most_common():
            print(f"  - {class_name}: {count}개")

    print("\n✨ 1단계 (다중 클래스) 완료! 이 마스크를 사용하여 2단계를 진행하세요.")

2단계 (지금 실행해야 할 코드):

데이터 포장 단계. 1단계에서 만든 원본 이미지와 마스크 이미지의 경로를 읽어와, 딥러닝 라이브러리(Hugging Face datasets)가 이해할 수 있는 형태의 **데이터셋(DatasetDict)**으로 만들어 줍니다.

이 데이터셋은 훈련용과 테스트용으로 나뉘게 됩니다.

In [ ]:
# 1단계 (수정): 특정 클래스만 선택하여 다중 클래스 마스크 생성

# 기본 라이브러리 임포트
import os
import json
import numpy as np
import cv2
from collections import Counter

print("라이브러리 로드 완료!")

# --- 경로 설정 ---
# ※ 이전 단계에서 확인된 경로를 사용합니다.
LABEL_DIR = '/content/sample_data/labels'

# 실제 이미지 폴더 경로를 자동으로 찾습니다.
possible_image_paths = [
    '/content/sample_data/JPEGImages_mosaic',
    '/content/sample_data/MonoCameraSemanticSegmentation/JPEGImages_mosaic',
    '/content/sample_data/MonoCameraSemanticSegmentation/JPEGImages',
]
IMAGE_DIR = None
for path in possible_image_paths:
    if os.path.exists(path):
        IMAGE_DIR = path
        break

# 새 마스크를 저장할 폴더
MASK_DIR_MULTI = '/content/sample_data/masks_multi_class'
os.makedirs(MASK_DIR_MULTI, exist_ok=True)

# --- 클래스 정의 ---
# 🎯 모델이 최종적으로 학습할 클래스와 고유 ID를 정의합니다.
# 이 맵은 나중에 모델 학습 시 id2label, label2id로 사용됩니다.
class_map = {
    "background": 0,
    "lane": 1,          # 모든 차선 관련 라벨은 'lane' 클래스(ID 1)로 통합됩니다.
    "bus": 2,
    "car": 3,
    "person": 4,
    "crosswalk": 5,
    "traffic_light": 6,
    "traffic_sign": 7,
    "pole": 8,
}

# 🎯 JSON 파일의 라벨 중 어떤 것을 사용할지, 그리고 어떤 클래스로 매핑할지 정의합니다.
target_labels = {
    # 차선 관련 라벨 -> 'lane' 클래스로 매핑
    'whdot': 'lane',
    'whsol': 'lane',
    'yesol': 'lane',
    'yedot': 'lane',
    'blsol': 'lane',
    'bldot': 'lane',
    'general road mark': 'lane',

    # 기타 객체 라벨 -> 해당 클래스로 직접 매핑
    'bus': 'bus',
    'car': 'car',
    'person': 'person',
    'crosswalk': 'crosswalk',
    'traffic_light': 'traffic_light',
    'traffic_sign': 'traffic_sign',
    'pole': 'pole',
}

print("✅ 클래스 및 경로 설정 완료")
print(f"  - 최종 클래스 수: {len(class_map)}개")
print(f"  - 마스크 저장 위치: {MASK_DIR_MULTI}")

# --- 데이터 매칭 ---
if not IMAGE_DIR or not os.path.exists(IMAGE_DIR):
    print("❌ 이미지 폴더를 찾을 수 없습니다! 이전 단계의 압축 해제가 정상적으로 완료되었는지 확인하세요.")
else:
    all_images = [f for f in os.listdir(IMAGE_DIR) if f.endswith('.png')]
    all_labels = [f for f in os.listdir(LABEL_DIR) if f.endswith('.json')]

    image_basenames = {f.replace('_leftImg8bit.png', '') for f in all_images}
    label_basenames = {f.replace('_gtFine_polygons.json', '') for f in all_labels}

    matched_basenames = sorted(list(image_basenames & label_basenames))
    print(f"🎯 처리할 총 데이터: {len(matched_basenames)}개")

    # --- 다중 클래스 마스크 생성 시작 ---
    print("\n🚀 다중 클래스 마스크 생성 시작...")
    processed_count = 0
    error_count = 0
    class_counts = Counter() # 각 클래스가 몇 번 그려졌는지 카운트

    # 매칭된 데이터만 처리
    for i, base_name in enumerate(matched_basenames):
        json_file = f"{base_name}_gtFine_polygons.json"
        image_file = f"{base_name}_leftImg8bit.png"

        label_path = os.path.join(LABEL_DIR, json_file)
        image_path = os.path.join(IMAGE_DIR, image_file)

        image = cv2.imread(image_path)
        if image is None:
            error_count += 1
            continue

        height, width, _ = image.shape
        mask = np.zeros((height, width), dtype=np.uint8) # 0 (background)으로 초기화

        try:
            with open(label_path, 'r') as f:
                data = json.load(f)
        except Exception as e:
            error_count += 1
            continue

        # JSON 파일의 각 객체에 대해 반복
        for obj in data['objects']:
            label_from_json = obj['label']

            # 이 라벨이 우리가 타겟으로 하는 라벨인지 확인
            if label_from_json in target_labels:

                # 최종 클래스 이름 가져오기 (예: 'whdot' -> 'lane')
                target_class_name = target_labels[label_from_json]

                # 클래스 ID 가져오기 (예: 'lane' -> 1)
                class_id = class_map[target_class_name]

                # 폴리곤 좌표 가져오기
                points = np.array(obj['polygon'], dtype=np.int32)

                # 마스크에 해당 클래스 ID로 폴리곤 그리기
                cv2.fillPoly(mask, [points], color=class_id)

                # 통계용 카운트
                class_counts[target_class_name] += 1

        # 마스크 파일 저장
        mask_file_name = image_file.replace('.png', '_mask_multi.png')
        mask_save_path = os.path.join(MASK_DIR_MULTI, mask_file_name)
        cv2.imwrite(mask_save_path, mask)

        processed_count += 1
        if i % 100 == 0: # 100개마다 진행 상황 출력
             print(f"  ✅ {i+1}/{len(matched_basenames)}: {mask_file_name} 생성 완료")

    print(f"\n🎉 다중 클래스 마스크 생성 완료!")
    print(f"  - ✅ 성공: {processed_count}개")
    print(f"  - ❌ 실패: {error_count}개")
    print(f"  - 📁 저장 위치: {MASK_DIR_MULTI}")

    # 생성된 파일 수 확인
    mask_files = [f for f in os.listdir(MASK_DIR_MULTI) if f.endswith('_mask_multi.png')]
    print(f"  - 📊 생성된 마스크 파일 수: {len(mask_files)}개")

    print("\n📊 처리된 클래스별 객체 수:")
    if not class_counts:
        print("  - 처리된 객체가 없습니다.")
    else:
        for class_name, count in class_counts.most_common():
            print(f"  - {class_name}: {count}개")

    print("\n✨ 1단계 (다중 클래스) 완료! 이 마스크를 사용하여 2단계를 진행하세요.")

🚀 3단계: 모델 학습 준비 (전처리)

ImageProcessor 불러오기: 사용할 모델(예: SegFormer)에 맞는 ImageProcessor를 불러옵니다.

이 도구는 이미지 크기를 조절하고, 색상 값을 정규화하는 등 모델의 입력 형식에 맞게 이미지를 자동으로 처리해줍니다.

전처리 함수 정의:
ImageProcessor를 사용하여 원본 이미지와 마스크를 변환하는 함수를 만듭니다.

이 단계에서 이미지 좌우 반전과 같은 데이터 증강(Data Augmentation) 기법을 적용하여 모델 성능을 높일 수 있습니다.

데이터셋에 전처리 적용: 위에서 만든 ds 데이터셋 전체에 전처리 함수를 적용합니다.

In [ ]:
# 3단계: 모델 학습 준비 (데이터 전처리) - 수정된 버전

# 필요한 라이브러리 설치
!pip install -q evaluate transformers "datasets>=2.14.0"

# 라이브러리 임포트
from transformers import SegformerImageProcessor
from torchvision.transforms import ColorJitter
import numpy as np

# --- 1. ImageProcessor 불러오기 ---
# 사용할 모델(SegFormer)에 맞는 전용 전처리 도구를 불러옵니다.
# 'do_reduce_labels=False'는 우리가 만든 마스크의 라벨 값을 그대로 사용하겠다는 의미입니다.
try:
    image_processor = SegformerImage_processor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512", do_reduce_labels=False)
except NameError:
    # 오타 수정
    SegformerImageProcessor = SegformerImageProcessor
    image_processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512", do_reduce_labels=False)


# 데이터 증강을 위한 도구
jitter = ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.1)

print("✅ ImageProcessor 로드 완료!")

# --- 2. 전처리 함수 정의 ---

# 훈련 데이터용 전처리 및 증강 함수
def train_transforms(example_batch):
    images = [jitter(x.convert("RGB")) for x in example_batch['image']]
    labels = [x for x in example_batch['label']]
    inputs = image_processor(images, labels, return_tensors="pt")
    return inputs

# 검증 및 테스트 데이터용 전처리 함수 (증강 없음)
def val_transforms(example_batch):
    images = [x.convert("RGB") for x in example_batch['image']]
    labels = [x for x in example_batch['label']]
    inputs = image_processor(images, labels, return_tensors="pt")
    return inputs

print("✅ 전처리 함수 정의 완료!")

# --- 3. 데이터셋에 전처리 함수 적용 (오류 수정된 부분) ---
# 각 데이터셋 스플릿에 개별적으로 접근하여 transform을 설정합니다.
try:
    ds['train'].set_transform(train_transforms)

    # 만약 2단계에서 'validation' 셋을 만들었다면 아래 줄의 주석을 해제하세요.
    if 'validation' in ds:
        ds['validation'].set_transform(val_transforms)

    ds['test'].set_transform(val_transforms)

    print("\n✅ 데이터셋에 전처리 적용 완료!")
    print(f"\n데이터셋 구조:")
    print(ds)

    print("\n✨ 3단계 완료! 데이터가 모델에 들어갈 모든 준비를 마쳤습니다.")
    print("이제 다음 4단계에서 모델을 불러올 차례입니다.")

except NameError:
    print("\n❌ 'ds' 데이터셋을 찾을 수 없습니다!")
    print("이전 2단계 코드를 먼저 실행하여 'ds' 변수를 생성해주세요.")
except KeyError as e:
    print(f"\n❌ 데이터셋에 '{e.args[0]}' 스플릿이 없습니다. 2단계에서 생성된 데이터셋을 확인해주세요.")

🚀 4단계: 모델 불러오기 및 설정

✅ 모델 로드 및 설정 완료!
  - 모델 아키텍처: segformer
  - 예측할 클래스 수: 9
  
이제 학습시킬 주인공인 딥러닝 모델을 불러옵니다.

사전 학습된 모델 불러오기:
 Hugging Face Hub에서 이미 대규모 데이터로 학습된 모델(Pre-trained model, 예: nvidia/segformer-b0-finetuned-ade-512-512)을 불러옵니다.

 처음부터 만드는 것보다 훨씬 빠르고 성능이 좋습니다.

모델 헤드(Head) 교체: 불러온 모델은 원래 다른 데이터셋(예: 20개의 클래스)에 맞춰져 있습니다.

모델의 마지막 출력 부분(분류기 헤드)을 우리가 정의한 9개의 클래스(id2label 기준)에 맞게 새로 설정해줘야 합니다.

In [ ]:
# 4단계: 모델 불러오기 및 설정

# 필요한 라이브러리 임포트
from transformers import SegformerForSemanticSegmentation

# --- 모델 불러오기 및 설정 ---
# 이전에 정의했던 클래스 정보를 다시 한번 확인합니다.
# 이 정보는 모델의 최종 출력층을 설정하는 데 필수적입니다.
try:
    # 2단계에서 정의한 id2label, label2id가 이미 메모리에 있다고 가정합니다.
    print("기존에 정의된 클래스 정보 사용:")
    print(f"  - 라벨 수: {len(id2label)}")
    print(f"  - 라벨 목록: {list(id2label.values())}")
except NameError:
    # 만약 세션이 끊겨서 변수가 사라졌을 경우를 대비해 다시 정의합니다.
    print("클래스 정보를 다시 정의합니다...")
    id2label = {
        0: "background", 1: "lane", 2: "bus", 3: "car", 4: "person",
        5: "crosswalk", 6: "traffic_light", 7: "traffic_sign", 8: "pole"
    }
    label2id = {v: k for k, v in id2label.items()}
    print(f"  - 라벨 수: {len(id2label)}")


# --- 사전 학습된 SegFormer 모델 불러오기 ---
# Hugging Face Hub에서 'nvidia/segformer-b0-finetuned-ade-512-512' 모델을 불러옵니다.
# 이 모델은 작고 효율적이면서도 좋은 성능을 내는 것으로 알려져 있습니다.
# 'num_labels'와 id2label, label2id를 우리 데이터에 맞게 설정해주면,
# 라이브러리가 알아서 모델의 마지막 부분을 교체해줍니다.
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512",
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True, # 크기가 다른 기존 출력층은 무시하고 새로 만듦
)

print("\n✅ 모델 로드 및 설정 완료!")
print(f"  - 모델 아키텍처: {model.config.model_type}")
print(f"  - 예측할 클래스 수: {model.config.num_labels}")

# 모델의 구조를 간단히 출력하여 확인할 수 있습니다.
# print(model)

print("\n✨ 4단계 완료! 이제 이 모델을 훈련시킬 준비가 되었습니다.")
print("다음 5단계에서 훈련 규칙을 정하고 실제 학습을 시작합니다.")

🚀 5단계: 훈련 설정 및 실행

모델 훈련을 위한 세부 규칙(하이퍼파라미터)을 정하고 훈련을 시작합니다.

TrainingArguments 설정: 학습률, 에포크(전체 데이터 반복 횟수), 배치 사이즈(한 번에 처리할 데이터 양), 모델 저장 경로 등 훈련에 관한 모든 규칙을 TrainingArguments 객체에 정의합니다.

Trainer 생성: Trainer 객체에 모델, 훈련 규칙, 훈련 데이터셋, 검증 데이터셋을 모두 전달하여 훈련 준비를 마칩니다.

훈련 시작: trainer.train() 명령어를 실행하면, GPU를 사용하여 본격적인 모델 학습이 시작됩니다.

학습 과정에서 매 에포크가 끝날 때마다 검증 데이터셋으로 모델 성능을 자동으로 평가하고 로그를 출력합니다.

In [ ]:
# 5단계: 모델 훈련 설정 및 실행 - 수정된 버전

# --- 0. 라이브러리 업그레이드 ---
# 이 명령어를 통해 라이브러리를 최신 버전으로 업데이트하여 'evaluation_strategy' 오류를 해결합니다.
!pip install -q -U transformers datasets evaluate

# 필요한 라이브러리 임포트
from transformers import TrainingArguments, Trainer
import torch
import numpy as np
import evaluate
import os

# Colab 환경에서 GPU 메모리 상태 확인 (옵션)
!nvidia-smi

# --- 1. 평가 지표 설정 ---
metric = evaluate.load("mean_iou")

def compute_metrics(eval_pred):
    with torch.no_grad():
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=1)

        metrics = metric.compute(
            predictions=predictions.flatten(),
            references=labels.flatten(),
            num_labels=len(id2label),
            ignore_index=0,
            reduce_labels=False,
        )

        for key, value in metrics.items():
            if isinstance(value, np.ndarray):
                class_iou = {id2label[i]: v for i, v in enumerate(value) if i in id2label}
                metrics[key] = class_iou

        return metrics

print("✅ 평가 지표 설정 완료!")

# --- 2. 훈련 규칙 (Hyperparameters) 정의 ---
# 이제 최신 버전의 라이브러리에서 모든 파라미터가 정상적으로 작동합니다.
training_args = TrainingArguments(
    output_dir="segformer-finetuned-korean-driving",
    learning_rate=6e-5,
    num_train_epochs=50,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    save_total_limit=3,
    evaluation_strategy="steps",   # 최신 버전에서는 이 파라미터가 정상적으로 인식됩니다.
    save_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=10,
    remove_unused_columns=False,
    push_to_hub=False,
    load_best_model_at_end=True,
)

print("✅ 훈련 규칙 정의 완료!")

# --- 3. Trainer 생성 및 훈련 시작 ---
# Trainer에 모델, 훈련 규칙, 데이터셋, 평가 함수를 모두 전달합니다.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    compute_metrics=compute_metrics,
)

print("\n🚀 모델 훈련을 시작합니다... (GPU 성능에 따라 시간이 소요됩니다)")

# 실제 훈련 시작
trainer.train()

print("\n🎉 모델 훈련 완료!")
print("✨ 5단계 완료! 이제 훈련된 모델의 최종 성능을 평가할 차례입니다.")

🚀 6단계: 평가 및 추론

훈련이 완료된 후 모델의 최
종 성능을 확인하고, 새로운 이미지에 사용해 봅니다.

최종 평가: trainer.evaluate(ds['test'])를 실행하여 테스트 데이터셋에 대한 최종 성능(점수)을 확인합니다.

추론(Inference): 학습된 모델을 사용하여 완전히 새로운 도로 이미지 한 장을 입력하고, 모델이 차선, 자동차, 사람 등을 얼마나 잘 예측하는지 시각적으로 확인합니다.